# 13 - The SQL Gauntlet: Intent- And Column-Aware Validation

`validate_query_context` does not grade SQL on vibes. Participation is
precise: COLUMN rows join the verdict only when the column is actually
referenced (or a star pulls everything), TABLE rows only when the
stated intent matches their scenario, and always-applicable read
controls apply to any SQL. This notebook runs real query shapes
through that model — JOINs, `SELECT *`, CTEs, a join into an
ungoverned table, byte-identical SQL under two intents, and a
cross-schema join.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None, schema="public"):
    ref = {"database": "acmecloud_demo", "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## 1. A JOIN is evaluated per referenced table

Both sides of the join get their own finding; the verdict aggregates
over every participating row.


In [ ]:
joined = client.validate_query_context(
    "SELECT c.region, SUM(s.arr) FROM customers c JOIN subscriptions s ON s.customer_id = c.customer_id GROUP BY c.region",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"customers JOIN subscriptions -> {joined['verdict']}")
for finding in joined["findings"]:
    print(f"  {finding['ref']['table']}: {finding['status']} ({finding.get('decision')})")


## 2. `SELECT *` pulls every masked column into the verdict

The star is a projection signal: it makes ALL column rows participate.
Name only the columns you need and the same table can pass clean.


In [ ]:
star = client.validate_query_context(
    "SELECT * FROM payment_methods LIMIT 5",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"SELECT *                    -> {star['verdict']} (both tokenized card columns participate)")

narrow = client.validate_query_context(
    "SELECT payment_method_id FROM payment_methods",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"SELECT payment_method_id    -> {narrow['verdict']} (no masked column referenced)")


## 3. CTE names are not tables

`recent` is a WITH-alias — the parser excludes it, so the only
finding is the real `subscriptions` reference.


In [ ]:
cte = client.validate_query_context(
    "WITH recent AS (SELECT customer_id, arr FROM subscriptions WHERE end_date IS NULL) SELECT customer_id, SUM(arr) FROM recent GROUP BY customer_id",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"CTE aggregate -> {cte['verdict']}")
print(f"findings: {[finding['ref']['table'] for finding in cte['findings']]}")


## 4. Joining an ungoverned table is called out, per ref

`legacy_customer_backup` is cataloged but ungoverned. The join still
gets an overall verdict — with an explicit per-ref
`not_enough_published_state` finding instead of a silent pass.


In [ ]:
legacy = client.validate_query_context(
    "SELECT c.customer_name, l.exported_at FROM customers c JOIN legacy_customer_backup l ON l.customer_id = c.customer_id",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"governed JOIN ungoverned -> {legacy['verdict']}")
for finding in legacy["findings"]:
    print(f"  {finding['ref']['table']}: {finding['status']} ({finding.get('reason_code')})")


## 5. The intent flip: same SQL, opposite verdicts

Byte-identical SQL. Under an analytics intent the permitted-use row
participates and the query passes; under a marketing intent the
prohibited-use deny participates and the same query fails. Intent is
part of the question — not an afterthought.


In [ ]:
analytics = client.validate_query_context(
    "SELECT region, COUNT(*) FROM customers GROUP BY region",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
marketing = client.validate_query_context(
    "SELECT region, COUNT(*) FROM customers GROUP BY region",
    scenario_key="purpose.prohibited_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"analytics intent -> {analytics['verdict']}")
print(f"marketing intent -> {marketing['verdict']} (same SQL, different question)")


## 6. Cross-schema joins resolve like anything else

Two-part names qualify against the default database — the `finance`
schema's guardrails answer for their own tables.


In [ ]:
finance = client.validate_query_context(
    "SELECT i.invoice_id, r.amount FROM finance.invoices i JOIN finance.revenue_ledger r ON r.invoice_id = i.invoice_id",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"finance.invoices JOIN finance.revenue_ledger -> {finance['verdict']}")
for finding in finance["findings"]:
    print(f"  {finding['ref']['schema']}.{finding['ref']['table']}: {finding.get('decision')}")
